In [3]:
import numpy as np
import pandas as pd
import seaborn as sns

In [4]:
df = pd.read_csv("kaggle_dataset.csv",encoding="utf-8-sig")
df.head()

,Unnamed: 0,title,date,stock
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,3.0,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,4.0,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400469 entries, 0 to 1400468
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   Unnamed: 0  1399180 non-null  float64
 1   title       1400469 non-null  object 
 2   date        1399180 non-null  object 
 3   stock       1397891 non-null  object 
dtypes: float64(1), object(3)
memory usage: 42.7+ MB


In [4]:
df.describe()

,Unnamed: 0
count,1.399180e+06
mean,7.095738e+05
std,4.077075e+05
min,0.000000e+00
25%,3.547798e+05
50%,7.114675e+05
75%,1.062727e+06
max,1.413848e+06


In [6]:
t = df.groupby("stock")
t.head()

,Unnamed: 0,title,date,stock
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,3.0,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,4.0,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A
...,...,...,...,...
1400402,1413782.0,China Zenix Auto International Announces Inten...,2018-06-15 09:01:00-04:00,ZX
1400403,1413783.0,China Zenix Auto Shares Halted News Pending,2018-06-13 16:52:00-04:00,ZX
1400404,1413784.0,"China Zenix Auto Q1 EPS $0.08, Made $130.123M ...",2018-05-17 06:01:00-04:00,ZX
1400405,1413785.0,"China Zenix Auto Reports Q4 EPS $0.03, Sales $...",2018-03-15 06:01:00-04:00,ZX


In [7]:
df_clean = df.dropna(subset = ["stock","title"])
stock_counts = df_clean['stock'].value_counts()
popular_stocks = stock_counts[stock_counts >= 1000].index
df = df_clean[df_clean['stock'].isin(popular_stocks)].copy()

In [8]:
import nltk
import string

In [9]:
from nltk.corpus import stopwords

In [10]:
english_stopwords = set(stopwords.words('english'))

def fn1(mess):
    clean = [char.lower() for char in mess if char not in string.punctuation and char not in "1234567890"]
    clean = ''.join(clean)
    
    return [word for word in clean.split() if word.lower() not in english_stopwords]
    

In [11]:
df['title'] = df['title'].astype(str).str.replace('\ufeff', '', regex=False)

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
b_transformer = CountVectorizer(analyzer = fn1,min_df = 1).fit(df_clean["title"])
vocab_list = b_transformer.get_feature_names_out()
title_bow = b_transformer.transform(df["title"])

In [15]:
from sklearn.feature_extraction.text import TfidfTransformer

In [16]:
tfidf_transformer = TfidfTransformer().fit(title_bow)
title_tfidf = tfidf_transformer.transform(title_bow)

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score
from sklearn.preprocessing import LabelEncoder

In [18]:
stock_detect_model = MultinomialNB()
X = title_tfidf
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(df["stock"])
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.3, random_state=42)
stock_detect_model.fit(X_train,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [19]:
pred = stock_detect_model.predict(X_test)

In [20]:
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score
print("The accuracy score is :",accuracy_score(y_test,pred))
print("The classification report is:",classification_report(y_test,pred))
print("The confusion matrix is:",confusion_matrix(y_test,pred))

The accuracy score is : 0.42486531004426753
The classification report is:               precision    recall  f1-score   support

           0       0.68      0.38      0.48       565
           1       0.83      0.41      0.55       400
           2       0.73      0.46      0.57       617
           3       0.97      0.45      0.61       285
           4       0.96      0.43      0.59       315
           5       0.82      0.50      0.62       417
           6       0.66      0.25      0.37       390
           7       0.60      0.60      0.60       721
           8       0.85      0.40      0.55       510
           9       0.64      0.53      0.58       451
          10       0.96      0.30      0.46       325
          11       0.65      0.47      0.54       326
          12       0.80      0.46      0.58       430
          13       0.75      0.48      0.58       507
          14       0.70      0.46      0.56       576
          15       0.77      0.23      0.36       505
       

E:\Anaconda Darun\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
E:\Anaconda Darun\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
E:\Anaconda Darun\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [19]:
# Frontend
def predict_new_headline(headline):
    cleaned_headline = ' '.join(fn1(headline))
    bow_vector = b_transformer.transform([cleaned_headline])
    tfidf_vector = tfidf_transformer.transform(bow_vector)
    predicted_encoded = stock_detect_model.predict(tfidf_vector)

    predicted_stock = label_encoder.inverse_transform(predicted_encoded)[0]
    
    print(f" Headline: '{headline}'")
    print(f" Predicted Stock: {predicted_stock}\n")

predict_new_headline("iPhone sales surge unexpected in Asian markets during Q3")
predict_new_headline("New electric vehicle factory opens in Texas with automated assembly lines")
predict_new_headline("Central bank increases interest rates by 25 basis points to combat inflation")

 Headline: 'iPhone sales surge unexpected in Asian markets during Q3'
 Predicted Stock: VZ

 Headline: 'New electric vehicle factory opens in Texas with automated assembly lines'
 Predicted Stock: TM

 Headline: 'Central bank increases interest rates by 25 basis points to combat inflation'
 Predicted Stock: EWU



In [27]:
# 1. Isolate the data the model actually trained on for Verizon (VZ)
vz_headlines = df[df['stock'] == 'VZ']['title']

# 2. Count how many times key terms show up in those headlines
iphone_count = vz_headlines.str.contains('iphone', case=False).sum()
sales_count = vz_headlines.str.contains('sales', case=False).sum()
surge_count = vz_headlines.str.contains('surge', case=False).sum()

print(f"📊 Out of your total VZ headlines:")
print(f"🔹 'iphone' appears: {iphone_count} times")
print(f"🔹 'sales' appears: {sales_count} times")
print(f"🔹 'surge' appears: {surge_count} times")

📊 Out of your total VZ headlines:
🔹 'iphone' appears: 83 times
🔹 'sales' appears: 54 times
🔹 'surge' appears: 12 times


In [28]:
!pip install streamlit joblib

  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0


In [21]:
import joblib
joblib.dump(stock_detect_model, 'stock_predictor_model.pkl')
joblib.dump(b_transformer, 'bag_of_words_transformer.pkl')
joblib.dump(tfidf_transformer, 'tfidf_weight_transformer.pkl')
joblib.dump(label_encoder, 'ticker_label_encoder.pkl')

['ticker_label_encoder.pkl']